# JS15 – Pipeline Machine Learning & Deployment
## End-to-End Pipeline, Model Serialization, dan Web Deployment pada Dataset Burnout

---

### Deskripsi & Tujuan
Pada modul **JS15**, mahasiswa mempelajari alur lengkap *Machine Learning Operations* (MLOps):
1. **Pipeline Terpadu**: Menggabungkan prapemrosesan data menggunakan Scikit-Learn `ColumnTransformer` dan `Pipeline` untuk penanganan nilai hilang, standarisasi fitur, dan *one-hot encoding*.
2. **Serialisasi Model & Pipeline**: Menyimpan model deep learning (`.h5`) dan objek transformer (`.pkl`) agar inferensi di server produksi konsisten 100% dengan tahap pelatihan.
3. **Web Server Deployment (Flask & Docker)**: Mengintegrasikan model ke dalam aplikasi web interaktif, dikemas dalam container Docker, dan siap dideploy ke layanan cloud publik seperti **Hugging Face Spaces**.


### 1. Import Library & Persiapan Data

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(42)
np.random.seed(42)

# Muat dataset burnout
df = pd.read_csv('burnout_dataset1.csv')
df_clean = df.dropna(subset=['Burnout_Risk']).copy()

X = df_clean.drop(columns=['Person_ID', 'Burnout_Risk', 'Burnout_Score', 'AI_Wellness_Recommendation'])
risk_map = {'Low': 0, 'Moderate': 1, 'High': 2}
risk_map_inv = {0: 'Low (Rendah)', 1: 'Moderate (Sedang)', 2: 'High (Tinggi)'}
y = df_clean['Burnout_Risk'].map(risk_map).values
target_names = ['Low', 'Moderate', 'High']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Fitur Numerik    : {len(num_cols)} kolom")
print(f"Fitur Kategorikal: {len(cat_cols)} kolom")


Fitur Numerik    : 19 kolom
Fitur Kategorikal: 17 kolom


### 2. Membangun & Memvalidasi Preprocessing Pipeline Scikit-Learn

In [2]:
# Pipeline preprocessing terstandarisasi
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Split data latih dan uji
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Fit pipeline pada X_train dan transform
X_train = preprocessor.fit_transform(X_train_raw)
X_test  = preprocessor.transform(X_test_raw)

y_train_cat = to_categorical(y_train, 3)
y_test_cat  = to_categorical(y_test, 3)

feature_dim = X_train.shape[1]
print(f"Dimensi fitur setelah pipeline: {feature_dim}")


Dimensi fitur setelah pipeline: 109


### 3. Pelatihan Model Klasifikasi Burnout untuk Deployment

In [3]:
model = models.Sequential([
    layers.Input(shape=(feature_dim,)),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

history = model.fit(
    X_train, y_train_cat,
    validation_split=0.15,
    epochs=35,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nAkurasi Model Siap Deploy: {test_acc * 100:.2f}%")


Epoch 1/35


  1/101 ━━━━━━━━━━━━━━━━━━━━ 2:33 2s/step - accuracy: 0.3281 - loss: 1.4834

 17/101 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4404 - loss: 1.1992 

 31/101 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5074 - loss: 1.0623

 44/101 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5548 - loss: 0.9691

 59/101 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5961 - loss: 0.8867

... (60 outputs left out)

 59/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9886 - loss: 0.0351

 89/101 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9891 - loss: 0.0340

101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9908 - loss: 0.0314 - val_accuracy: 0.9850 - val_loss: 0.0659



Akurasi Model Siap Deploy: 99.05%


### 4. Ekspor Artefak Model & Pipeline (Penyimpanan File)

In [4]:
# 1. Simpan Model Neural Network (.h5)
model_file = 'burnout_model.h5'
model.save(model_file)
print(f"[OK] Model Keras disimpan: {model_file} ({os.path.getsize(model_file)/(1024*1024):.2f} MB)")

# 2. Simpan Pipeline Preprocessing & Skema Kolom (.pkl)
pipeline_file = 'burnout_preprocessor.pkl'
pipeline_data = {
    'preprocessor': preprocessor,
    'feature_names_num': num_cols,
    'feature_names_cat': cat_cols,
    'risk_map_inv': risk_map_inv
}

with open(pipeline_file, 'wb') as f:
    pickle.dump(pipeline_data, f)
print(f"[OK] Preprocessor disimpan: {pipeline_file} ({os.path.getsize(pipeline_file)/(1024):.2f} KB)")


[OK] Model Keras disimpan: burnout_model.h5 (0.34 MB)
[OK] Preprocessor disimpan: burnout_preprocessor.pkl (5.15 KB)


### 5. Pengujian Simulasi Inferensi Mandiri (End-to-End Sanity Test)

In [5]:
# Muat ulang dari disk untuk memverifikasi kesiapan server
loaded_model = tf.keras.models.load_model('burnout_model.h5')
with open('burnout_preprocessor.pkl', 'rb') as f:
    loaded_pipe = pickle.load(f)

# Buat contoh data responden baru (Karyawan simulasi)
sample_employee = pd.DataFrame([{
    'Age': 29,
    'Gender': 'Female',
    'Country': 'Indonesia',
    'Occupation': 'Software Engineer',
    'Education_Level': 'Bachelor',
    'Employment_Status': 'Full-time',
    'Monthly_Income_USD': 3000,
    'Work_Hours_Per_Week': 55,
    'Remote_Work': 'Hybrid',
    'Job_Satisfaction': 4.5,
    'Work_Life_Balance': 3.5,
    'Sleep_Hours': 5.5,
    'Sleep_Quality': 'Poor',
    'Stress_Level': 'High',
    'Anxiety_Score': 7.5,
    'Depression_Score': 6.0,
    'Mood_Score': 5.0,
    'Emotional_Stability': 4.0,
    'Physical_Activity_Hours': 1.0,
    'Exercise_Frequency': 'Rarely',
    'Meditation_Minutes': 0,
    'Screen_Time_Hours': 10.0,
    'Social_Media_Hours': 4.0,
    'Gaming_Hours': 1.0,
    'Coffee_Cups_Per_Day': 3,
    'Alcohol_Consumption': 'None',
    'Smoking': 'No',
    'Healthy_Diet': 'No',
    'Chronic_Stress': 'Yes',
    'Family_History_Mental_Illness': 'No',
    'Therapy_Attendance': 'No',
    'Support_System': 'Weak',
    'Life_Satisfaction': 4.5,
    'Productivity_Score': 5.5,
    'Absenteeism_Days': 8,
    'Mental_Health_Status': 'Stressed'
}])

# Transformasi dan Prediksi
sample_trans = loaded_pipe['preprocessor'].transform(sample_employee)
probs = loaded_model.predict(sample_trans, verbose=0)[0]
pred_class = int(np.argmax(probs))
pred_label = loaded_pipe['risk_map_inv'][pred_class]

print("=== HASIL UJI INFERENSI SAMPEL RESUS ===")
print(f"Hasil Prediksi Kategori : {pred_label}")
print(f"Probabilitas Low        : {probs[0]*100:.2f}%")
print(f"Probabilitas Moderate   : {probs[1]*100:.2f}%")
print(f"Probabilitas High       : {probs[2]*100:.2f}%")


=== HASIL UJI INFERENSI SAMPEL RESUS ===
Hasil Prediksi Kategori : High (Tinggi)
Probabilitas Low        : 0.03%
Probabilitas Moderate   : 4.43%
Probabilitas High       : 95.54%


### 6. Verifikasi Struktur Paket Deployment (`burnout-project/`)
Untuk deployment ke **Hugging Face Spaces** dengan Docker SDK, struktur direktori paket telah disusun:
```text
burnout-project/
├── app.py                     # Web Server Flask dengan GUI input form & hasil rekomendasi
├── Dockerfile                 # Konfigurasi container Linux Debian Python 3.9 Slim
├── requirements.txt           # Dependensi flask, tensorflow-cpu, pandas, scikit-learn
├── README.md                  # Metadata Hugging Face Spaces
├── burnout_model.h5           # Model neural network
└── burnout_preprocessor.pkl   # Objek preprocessor pipeline
```


In [6]:
# Verifikasi berkas deployment
deploy_folder = "burnout-project"
files_in_deploy = os.listdir(deploy_folder)
print("Berkas dalam paket deployment:")
for f in files_in_deploy:
    sz = os.path.getsize(os.path.join(deploy_folder, f))
    print(f" - {f:<25} ({sz:,} bytes)")


Berkas dalam paket deployment:
 - app.py                    (13,618 bytes)
 - burnout_model.h5          (355,616 bytes)
 - burnout_preprocessor.pkl  (5,461 bytes)
 - Dockerfile                (435 bytes)
 - README.md                 (308 bytes)
 - requirements.txt          (62 bytes)


### 7. Kesimpulan Praktikum JS15
1. **Pipeline Utuh (End-to-End)**: Penggabungan Scikit-Learn `ColumnTransformer` dan model Keras `Sequential` menjamin proses inferensi di lingkungan produksi mereplikasi secara akurat standarisasi dan imputasi data latihan.
2. **Containerization & Deployment**: Penggunaan container Docker pada Hugging Face Spaces memungkinkan aplikasi web Flask dijalankan di cloud publik dengan isolasi dependensi yang aman, stabil, dan dapat diakses dari mana saja.
